# Research Intelligence Desk

**Mode:** Offline  
**Level:** Capstone  
**Estimated time:** 60 minutes

A market-entry decision is not a summarization problem. It is an evidence
coordination problem: specialists must finish, claims need provenance, conflicts
must remain visible, and a skeptical review must be able to force revision.


## Problem and outcome

Northstar Analytics is considering the UK mid-market logistics sector. The
evidence packet mixes interviews, competitor claims, an internal pilot, one
stale analyst note, and two deliberate contradictions. We will build an
eight-agent desk that produces a cited decision brief and refuses to publish an
unsupported performance claim.

The final artifact includes a recommendation, counterarguments, uncertainties,
validation experiments, citations, and the revision trail that created it.


In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "examples" / "notebooks"]
support_dir = next(path for path in candidates if (path / "support.py").is_file())
if str(support_dir) not in sys.path:
    sys.path.insert(0, str(support_dir))

from support import (
    find_example_asset, setup_case_study, show_artifact, show_callout,
    show_flow, show_json, show_message_graph, show_roles, show_scorecard,
    show_spore, show_timeline, stage,
)

setup_case_study("Research Intelligence Desk")


## Course prerequisites

Complete the research pipeline, feedback loop, parallel-agent, tool, and memory
lessons first. This capstone assumes you recognize a Spore and know why Reef
completion is stronger than a timed sleep.


## Architecture and responsibilities

Every message uses the same contract: `type`, `domain_id`, `correlation_id`,
`producer`, `stage`, `status`, and `payload`. Specialists may report `partial`,
but they may not silently disappear.

```text
research_request → specialist_finding × 3 → evidence_audit → draft_brief
                 → review_decision → revised_brief → final_brief
```


In [ ]:
show_flow(
    ("Director", "one bounded request", "human"),
    ("Specialists", "parallel terminal findings", "agent"),
    ("Audit + review", "reject weak evidence", "tool"),
    ("Editor", "one bounded revision", "agent"),
    ("Publisher", "citation-complete brief", "spore"),
)
show_roles([
    ("Research director", "Own the question and final acceptance", "agent"),
    ("Market researcher", "Market structure and feasibility evidence", "agent"),
    ("Customer researcher", "Interview evidence and buying constraints", "agent"),
    ("Competitor researcher", "Pricing and vendor-claim evidence", "agent"),
    ("Evidence auditor", "Freshness, conflicts, and admissibility", "agent"),
    ("Skeptical reviewer", "Challenge claim-to-source support", "agent"),
    ("Editor", "Draft and revise the decision brief", "agent"),
    ("Publisher", "Enforce the final artifact contract", "agent"),
])


## Design decisions

- Evidence is data, not prose hidden in prompts.
- Every specialist emits a terminal result. A partial result carries rejected
  source IDs and a reason.
- Tools own source lookup, freshness, and citation coverage.
- The first draft is intentionally wrong: it turns a competitor's unverified
  claim into a Northstar performance claim.
- The project memory retains accepted evidence and both editorial versions.


## Implementation

We first make the message contract executable. This prevents a later agent from
dropping the correlation ID or inventing an ambiguous payload shape.


In [ ]:
import json
from datetime import date

from praval import Agent, agent, broadcast, get_reef, start_agents
from praval.core.reef import reset_reef
from praval.core.tool_registry import reset_tool_registry
from praval.tools import tool

reset_reef()
reset_tool_registry()

REQUIRED = {"type", "domain_id", "correlation_id", "producer", "stage", "status", "payload"}


def research_message(message_type, producer, stage_name, status, payload):
    message = {
        "type": message_type, "domain_id": "MARKET-UK-2026",
        "correlation_id": "research-uk-logistics-001", "producer": producer,
        "stage": stage_name, "status": status, "payload": payload,
    }
    assert set(message) == REQUIRED
    return message


message_trail, received_spores, tool_events = [], [], []
specialist_results, revisions, final_briefs = {}, [], []


In [ ]:
packet_path = find_example_asset("notebooks/fixtures/research_sources.json")
packet = json.loads(packet_path.read_text(encoding="utf-8"))
sources = {item["id"]: item for item in packet["sources"]}


@tool("research_source_lookup", description="Look up one evidence record by ID.",
      category="research", shared=True)
def source_lookup(source_id: str) -> dict:
    tool_events.append({"tool": "source_lookup", "source_id": source_id})
    if source_id not in sources:
        raise ValueError(f"unknown source: {source_id}")
    return dict(sources[source_id])


@tool("research_freshness", description="Check evidence age against a fixed date.",
      category="research", shared=True)
def freshness_check(source_id: str, as_of: str) -> dict:
    source = source_lookup.execute_as_tool(source_id)
    age = (date.fromisoformat(as_of) - date.fromisoformat(source["published"])).days
    result = {"source_id": source_id, "age_days": age, "fresh": age <= 365}
    tool_events.append({"tool": "freshness_check", **result})
    return result


@tool("research_citation_coverage", description="Validate citations on every claim.",
      category="research", shared=True)
def citation_coverage(claims: list, accepted_ids: list) -> dict:
    uncovered = [claim["text"] for claim in claims if not claim.get("citations")]
    invalid = sorted({citation for claim in claims for citation in claim.get("citations", [])
                      if citation not in accepted_ids})
    result = {"complete": not uncovered and not invalid,
              "uncovered": uncovered, "invalid": invalid}
    tool_events.append({"tool": "citation_coverage", **result})
    return result


project_memory = Agent(
    "research-project-memory", provider="notebook-lifecycle",
    model="notebook-lifecycle-model", memory_enabled=True,
    memory_config={"backend": "memory"},
)
assert project_memory.memory is not None


In [ ]:
def remember_spore(spore):
    received_spores.append(spore)
    message_trail.append(dict(spore.knowledge))


@agent("research-director", responds_to=["research_request", "final_brief"],
       auto_broadcast=False)
def research_director(spore):
    remember_spore(spore)
    if spore.knowledge["type"] == "final_brief":
        final_briefs.append(spore.knowledge["payload"])
        stage("research director", "accepted final brief", "ready", spore)


def terminal_finding(spore, producer, source_ids):
    remember_spore(spore)
    records, rejected = [], []
    for source_id in source_ids:
        record = source_lookup.execute_as_tool(source_id)
        check = freshness_check.execute_as_tool(source_id, packet["as_of"])
        (records if check["fresh"] else rejected).append({**record, **check})
    status = "partial" if rejected else "complete"
    specialist_results[producer] = status
    stage(producer, "terminal finding", status, spore)
    broadcast(research_message("specialist_finding", producer, "specialist",
                               status, {"evidence": records, "rejected": rejected}))


@agent("market-researcher", responds_to=["research_request"],
       tools=["research_source_lookup", "research_freshness"], auto_broadcast=False)
def market_researcher(spore):
    terminal_finding(spore, "market-researcher", ["OPS-001", "ANL-001"])


@agent("customer-researcher", responds_to=["research_request"],
       tools=["research_source_lookup", "research_freshness"], auto_broadcast=False)
def customer_researcher(spore):
    terminal_finding(spore, "customer-researcher", ["INT-001", "INT-002", "INT-003"])


In [ ]:
@agent("competitor-researcher", responds_to=["research_request"],
       tools=["research_source_lookup", "research_freshness"], auto_broadcast=False)
def competitor_researcher(spore):
    terminal_finding(spore, "competitor-researcher", ["COMP-001", "COMP-002"])


finding_bucket = {}


@agent("evidence-auditor", responds_to=["specialist_finding"], auto_broadcast=False)
def evidence_auditor(spore):
    remember_spore(spore)
    finding_bucket[spore.knowledge["producer"]] = spore.knowledge["payload"]
    if len(finding_bucket) != 3:
        return
    accepted = [item for result in finding_bucket.values() for item in result["evidence"]]
    rejected = [item for result in finding_bucket.values() for item in result["rejected"]]
    audit = {
        "accepted_ids": sorted(item["id"] for item in accepted),
        "rejected_ids": sorted(item["id"] for item in rejected),
        "contradictions": packet["contradictions"],
        "limitations": ["COMP-002 is a vendor claim without published methodology"],
    }
    for item in accepted:
        project_memory.remember(f"Accepted evidence {item['id']}: {item['claim']}", 0.8)
    stage("evidence auditor", "audit complete", f"{len(rejected)} rejected", spore)
    broadcast(research_message("evidence_audit", "evidence-auditor", "audit",
                               "complete", audit))


In [ ]:
@agent("brief-editor", responds_to=["evidence_audit", "review_decision"],
       auto_broadcast=False)
def brief_editor(spore):
    remember_spore(spore)
    if spore.knowledge["type"] == "evidence_audit":
        audit = spore.knowledge["payload"]
        draft = {
            "revision": 0,
            "recommendation": "Enter with a 90-day design-partner pilot.",
            "claims": [
                {"text": "Exception reconciliation is frequent manual work.", "citations": ["INT-001"]},
                {"text": "Northstar reduces resolution time by 40 percent.", "citations": ["COMP-002"]},
                {"text": "Pilot interest exists below GBP 60 per seat.", "citations": ["INT-002"]},
            ],
            "audit": audit,
        }
        revisions.append(draft)
        project_memory.remember("Draft revision 0 contains an unverified performance claim", 0.9)
        broadcast(research_message("draft_brief", "brief-editor", "draft", "needs_review", draft))
        return
    decision = spore.knowledge["payload"]
    revised = {
        "revision": 1,
        "recommendation": "Enter with a 90-day design-partner pilot priced below GBP 60 per seat.",
        "claims": [
            {"text": "Exception reconciliation is frequent manual work.", "citations": ["INT-001"]},
            {"text": "A three-carrier integration is technically feasible.", "citations": ["OPS-001"]},
            {"text": "Pilot interest exists below GBP 60 per seat.", "citations": ["INT-002"]},
        ],
        "counterarguments": ["Security and data residency can block larger prospects."],
        "uncertainties": ["Willingness to pilot does not establish retention or list-price tolerance."],
        "validation_experiments": ["Run five paid pilots", "Complete a UK data-residency review"],
        "removed_claims": decision["unsupported_claims"],
    }
    revisions.append(revised)
    project_memory.remember("Revision 1 removed the unsupported 40 percent claim", 1.0)
    broadcast(research_message("revised_brief", "brief-editor", "revision", "complete", revised))


@agent("skeptical-reviewer", responds_to=["draft_brief"], auto_broadcast=False)
def skeptical_reviewer(spore):
    remember_spore(spore)
    unsupported = [claim["text"] for claim in spore.knowledge["payload"]["claims"]
                   if "Northstar reduces" in claim["text"]]
    decision = {"decision": "revise", "unsupported_claims": unsupported,
                "reason": "A competitor vendor claim cannot establish Northstar performance."}
    assert unsupported
    stage("skeptical reviewer", "revision required", unsupported[0], spore)
    broadcast(research_message("review_decision", "skeptical-reviewer", "review",
                               "rejected", decision))


In [ ]:
@agent("brief-publisher", responds_to=["revised_brief"],
       tools=["research_citation_coverage"], auto_broadcast=False)
def brief_publisher(spore):
    remember_spore(spore)
    brief = spore.knowledge["payload"]
    accepted = finding_bucket["market-researcher"]["evidence"]
    accepted += finding_bucket["customer-researcher"]["evidence"]
    accepted += finding_bucket["competitor-researcher"]["evidence"]
    accepted_ids = sorted(item["id"] for item in accepted)
    coverage = citation_coverage.execute_as_tool(brief["claims"], accepted_ids)
    assert coverage["complete"]
    final = {
        "title": "UK logistics market-entry decision brief",
        **brief,
        "citation_coverage": coverage,
        "correlation_id": spore.knowledge["correlation_id"],
        "status": "publishable",
    }
    stage("publisher", "publication contract passed", "citation complete", spore)
    broadcast(research_message("final_brief", "brief-publisher", "publish",
                               "complete", final))


## End-to-end run

The initial broadcast fans out to three specialists. The rest of the workflow is
driven by messages emitted inside handlers. `wait_for_completion()` covers the
entire cascade, including the forced editorial revision.


In [ ]:
agents = (
    research_director, market_researcher, customer_researcher,
    competitor_researcher, evidence_auditor, brief_editor,
    skeptical_reviewer, brief_publisher,
)
start_agents(
    *agents,
    initial_data=research_message(
        "research_request", "research-director", "intake", "open",
        {"question": "Should Northstar enter UK mid-market logistics?"},
    ),
    channel="research-intelligence-desk",
)
reef = get_reef()
assert reef.wait_for_completion(timeout=30)

assert specialist_results == {
    "market-researcher": "partial",
    "customer-researcher": "complete",
    "competitor-researcher": "complete",
}
assert len(final_briefs) == 1 and len(revisions) == 2
assert final_briefs[0]["status"] == "publishable"
assert {item["correlation_id"] for item in message_trail} == {"research-uk-logistics-001"}


## Inspect the system

The useful output is not only the brief. The trail shows which agent produced
each state transition, while project memory shows what survived the handoffs.


In [ ]:
show_message_graph(message_trail)
show_spore(received_spores[0], "Actual opening Spore")
show_json(
    {
        "specialist_terminal_status": specialist_results,
        "rejected_evidence": revisions[0]["audit"]["rejected_ids"],
        "tool_executions": tool_events,
        "memory": [entry.content for entry in project_memory.recall("evidence revision", limit=20)],
    },
    "Evidence and execution state",
)
show_timeline()


In [ ]:
final = final_briefs[0]
assert "ANL-001" in revisions[0]["audit"]["rejected_ids"]
assert any("40 percent" in item for item in final["removed_claims"])
assert final["citation_coverage"]["complete"]
assert all(claim["citations"] for claim in final["claims"])

show_scorecard([
    ("Specialists terminal", "3 of 3", "pass"),
    ("Stale evidence rejected", "ANL-001", "pass"),
    ("Editorial revisions", len(revisions) - 1, "pass"),
    ("Citation coverage", "complete", "pass"),
    ("Correlations", len({item["correlation_id"] for item in message_trail}), "pass"),
])
show_artifact("Published decision brief", final)


## Failure modes and tradeoffs

This desk is deterministic, so it demonstrates coordination and evidence
discipline rather than open-ended research. In production, source acquisition
would be fallible and specialists would need retry budgets. The terminal
`partial` result is important: fan-in can distinguish an honest limitation from
a worker that never returned.


## Extensions

Try adding a regulatory researcher, a source-access failure, or a rule that two
independent sources must support pricing claims. Preserve the terminal-result
contract so the auditor always knows when fan-out has finished.

## Cleanup

Memory, decorated Agents, Reef workers, and the global tool registry are
explicitly released so the notebook remains safe to rerun.


In [ ]:
project_memory.memory.clear_agent_memories(project_memory.name)
project_memory.close()
for function in agents:
    function._praval_agent.close()
assert reef.shutdown(timeout=10)
reset_tool_registry()
show_callout("Research desk closed", "Agents, memory, Reef, and tools were released.", role="reef")
